# COMP3018 — US 10-Year Treasury Yield Feature
## Data Ingestion & Exploratory Analysis
**Student:** Tom [Surname] | **FAN:** [YourFAN]  
**Feature:** US 10-Year Treasury Yield (`^TNX` via yfinance / `DGS10` via FRED)  
**Causal Link:** Rising yields increase discount rates applied to energy company future cash flows,
suppress equity valuations, and signal USD strength — all of which affect ASX energy stock returns.

---
This notebook:
1. Fetches US 10-Year Treasury yield data via two methods (yfinance + FRED)
2. Performs exploratory data analysis (EDA)
3. Demonstrates why this feature exhibits non-trivial uncertainty
4. Outputs a clean CSV ready for the team pipeline


## 1. Imports & Configuration

In [ ]:
# Standard library
import warnings
warnings.filterwarnings("ignore")

# Data
import requests
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Stats
import scipy.stats as stats

# yfinance (primary method)
import yfinance as yf

# ── Configuration ──────────────────────────────────────────────────────────
START_DATE = "2019-01-01"   # 6 years — captures pre-COVID, COVID, hike cycle
END_DATE   = "2025-01-01"

OUTPUT_PATH = "../data/raw/us_10yr_yield.csv"

print(f"Date range: {START_DATE} → {END_DATE}")
print("Imports OK.")


: 

## 2. Data Ingestion

### Method A — yfinance (`^TNX`)
`^TNX` is the CBOE 10-Year Treasury Note yield index on Yahoo Finance.
Values are in **percent** (e.g. 4.25 = 4.25% yield).
This is the recommended method as it uses the same library as the rest of the team pipeline.


In [ ]:
def fetch_yfinance(start: str, end: str) -> pd.DataFrame:
    """
    Fetch US 10-Year Treasury yield via yfinance ticker ^TNX.

    Args:
        start: Start date string (YYYY-MM-DD).
        end:   End date string (YYYY-MM-DD).

    Returns:
        DataFrame with daily yield (Close) indexed by Date.
    """
    raw = yf.download("^TNX", start=start, end=end, auto_adjust=True, progress=False)

    if raw.empty:
        print("WARNING: yfinance returned no data. Try Method B (FRED) below.")
        return pd.DataFrame()

    # Extract Close (daily closing yield) and flatten MultiIndex if present
    if isinstance(raw.columns, pd.MultiIndex):
        df = raw["Close"].rename(columns={"^TNX": "yield_pct"})
    else:
        df = raw[["Close"]].rename(columns={"Close": "yield_pct"})

    df.index.name = "Date"
    df.index = pd.to_datetime(df.index)
    print(f"yfinance: fetched {len(df)} trading days  |  "
          f"range: {df.index[0].date()} → {df.index[-1].date()}")
    return df


yield_df = fetch_yfinance(START_DATE, END_DATE)
yield_df.head(10)


### Method B — FRED Direct API (`DGS10`)
Fallback if yfinance is unavailable. FRED is the Federal Reserve's official data service.
No API key is required for public series. Values are also in percent.


In [ ]:
def fetch_fred(start: str, end: str, api_key: str = None) -> pd.DataFrame:
    """
    Fetch DGS10 (US 10-Year Treasury yield) directly from the FRED REST API.

    A free API key can be obtained at https://fred.stlouisfed.org/docs/api/api_key.html
    Without a key, the request is still accepted for public series but may be
    rate-limited. Pass api_key= or set FRED_API_KEY environment variable.

    Args:
        start:   Start date string (YYYY-MM-DD).
        end:     End date string (YYYY-MM-DD).
        api_key: Optional FRED API key string.

    Returns:
        DataFrame with daily yield indexed by Date, or empty DataFrame on failure.
    """
    import os
    key = api_key or os.environ.get("FRED_API_KEY", "")

    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id":          "DGS10",
        "observation_start":  start,
        "observation_end":    end,
        "file_type":          "json",
    }
    if key:
        params["api_key"] = key

    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"FRED request failed: {e}")
        return pd.DataFrame()

    observations = resp.json().get("observations", [])
    if not observations:
        print("FRED returned no observations.")
        return pd.DataFrame()

    df = pd.DataFrame(observations)[["date", "value"]]
    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")  # "." → NaN on holidays
    df = df.rename(columns={"date": "Date", "value": "yield_pct"}).set_index("Date")

    # FRED includes weekends/holidays as NaN — drop them
    df = df.dropna()
    print(f"FRED:     fetched {len(df)} trading days  |  "
          f"range: {df.index[0].date()} → {df.index[-1].date()}")
    return df


# Try FRED — use this if yfinance fails above
yield_df_fred = fetch_fred(START_DATE, END_DATE)

# Use whichever succeeded
if yield_df.empty and not yield_df_fred.empty:
    yield_df = yield_df_fred
    print("Using FRED data.")
elif not yield_df.empty:
    print("Using yfinance data.")
else:
    print("ERROR: Both methods failed. Check your internet connection.")


## 3. Data Validation

In [ ]:
print("=== Data Quality Report ===")
print(f"Shape:          {yield_df.shape}")
print(f"Date range:     {yield_df.index.min().date()} → {yield_df.index.max().date()}")
print(f"Missing values: {yield_df['yield_pct'].isna().sum()}")
print(f"Missing %:      {yield_df['yield_pct'].isna().mean()*100:.2f}%")
print()
print("=== Descriptive Statistics ===")
print(yield_df["yield_pct"].describe().round(4))

# Sanity check — 10yr yield should be between 0% and 15% in modern era
out_of_range = yield_df[(yield_df["yield_pct"] < 0) | (yield_df["yield_pct"] > 15)]
if not out_of_range.empty:
    print(f"\nWARNING: {len(out_of_range)} out-of-range values detected:")
    print(out_of_range)
else:
    print("\nRange check PASSED — all values within expected bounds (0–15%).")


## 4. Exploratory Data Analysis

### 4.1 Yield Level Over Time
This plot reveals the key **regime change** in the data — the shift from near-zero
yields (2020–2022) to the aggressive hiking cycle (2022–2024). This regime
non-stationarity is a critical source of uncertainty and motivates the Bayesian
approach: a frequentist model fit on the full period will average across regimes,
whereas a Bayesian model can encode prior beliefs about the current regime.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle("US 10-Year Treasury Yield (^TNX / DGS10) — EDA", fontsize=13, fontweight="bold")

# ── Panel 1: Yield level ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(yield_df.index, yield_df["yield_pct"], color="steelblue", lw=1.2)
ax.axhspan(0, 1.5, alpha=0.08, color="green", label="Near-zero rate regime")
ax.axhspan(3.5, 6,  alpha=0.08, color="red",   label="Hiking cycle regime")
ax.set_title("Daily Yield Level (%)")
ax.set_ylabel("Yield (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Panel 2: Daily change in yield ───────────────────────────────────────
yield_df["daily_change"] = yield_df["yield_pct"].diff()
ax = axes[1]
ax.bar(yield_df.index, yield_df["daily_change"],
       color=["tomato" if x < 0 else "steelblue" for x in yield_df["daily_change"]],
       width=1, alpha=0.7)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Daily Change in Yield (bp equivalent)")
ax.set_ylabel("Change (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(alpha=0.3)

# ── Panel 3: 21-day rolling volatility of yield changes ──────────────────
yield_df["vol_21d"] = yield_df["daily_change"].rolling(21).std()
ax = axes[2]
ax.fill_between(yield_df.index, yield_df["vol_21d"], color="darkorange", alpha=0.6)
ax.set_title("21-Day Rolling Volatility of Daily Yield Changes")
ax.set_ylabel("Std Dev (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../data/raw/yield_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → ../data/raw/yield_eda.png")


### 4.2 Distribution of Daily Yield Changes
This is the key input for the probabilistic estimation layer.
We test whether yield changes are Normally distributed — if they are not
(heavy tails, skewness), this directly justifies the Bayesian approach
and the use of a Student-t likelihood.


In [ ]:
changes = yield_df["daily_change"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Distribution of Daily Yield Changes", fontsize=12, fontweight="bold")

# ── Histogram with Normal fit ─────────────────────────────────────────────
ax = axes[0]
ax.hist(changes, bins=60, density=True, alpha=0.5, color="steelblue", label="Empirical")
x = np.linspace(changes.min(), changes.max(), 300)
mu_mle, sigma_mle = stats.norm.fit(changes)
ax.plot(x, stats.norm.pdf(x, mu_mle, sigma_mle),
        "r-", lw=2, label=f"Normal MLE\nμ={mu_mle:.4f}, σ={sigma_mle:.4f}")
df_t, loc_t, scale_t = stats.t.fit(changes)
ax.plot(x, stats.t.pdf(x, df_t, loc_t, scale_t),
        "g--", lw=2, label=f"Student-t MLE\ndf={df_t:.1f}")
ax.set_title("Histogram + MLE Fits")
ax.set_xlabel("Daily Change (%)")
ax.legend(fontsize=7)

# ── Q-Q plot ──────────────────────────────────────────────────────────────
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(changes, dist="norm")
ax.scatter(osm, osr, s=4, alpha=0.4, color="steelblue")
ax.plot(osm, slope * np.array(osm) + intercept, "r-", lw=1.5, label=f"R²={r**2:.4f}")
ax.set_title("Normal Q-Q Plot")
ax.set_xlabel("Theoretical Quantiles")
ax.set_ylabel("Sample Quantiles")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Summary statistics ────────────────────────────────────────────────────
ax = axes[2]
ax.axis("off")
skew     = float(changes.skew())
kurt     = float(changes.kurtosis())   # Excess kurtosis (Normal = 0)
ks_stat, ks_p = stats.kstest(changes, "norm", args=(mu_mle, sigma_mle))
sw_stat, sw_p = stats.shapiro(changes.sample(min(len(changes), 5000), random_state=42))

summary_text = (
    f"  n observations:    {len(changes)}\n"
    f"  Mean:              {changes.mean():.5f}%\n"
    f"  Std Dev:           {changes.std():.5f}%\n"
    f"  Skewness:          {skew:.4f}\n"
    f"  Excess Kurtosis:   {kurt:.4f}\n"
    f"  (Normal = 0; >0 = heavy tails)\n\n"
    f"  KS Test (Normal):\n"
    f"    statistic = {ks_stat:.4f}\n"
    f"    p-value   = {ks_p:.4f}\n"
    f"  {'REJECT normality' if ks_p < 0.05 else 'Cannot reject normality'} at α=0.05\n\n"
    f"  Shapiro-Wilk Test:\n"
    f"    statistic = {sw_stat:.4f}\n"
    f"    p-value   = {sw_p:.4f}\n"
    f"  {'REJECT normality' if sw_p < 0.05 else 'Cannot reject normality'} at α=0.05"
)
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
ax.set_title("Normality Tests")

plt.tight_layout()
plt.savefig("../data/raw/yield_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Key insight for report
print("\n=== KEY INSIGHT FOR REPORT ===")
if ks_p < 0.05 or sw_p < 0.05:
    print(f"Normality REJECTED — excess kurtosis = {kurt:.2f}")
    print("Daily yield changes have heavier tails than a Normal distribution.")
    print("This justifies using a Student-t likelihood in both MLE and Bayesian models,")
    print("and demonstrates why a point estimate (MLE) understates tail risk.")
else:
    print("Normality not rejected at α=0.05.")


## 5. Export Clean Data for Pipeline

In [ ]:
import os, pathlib

# Build final clean feature DataFrame
output = yield_df[["yield_pct", "daily_change", "vol_21d"]].copy()

# Forward-fill any gaps (e.g. US public holidays not in AU calendar)
output = output.ffill()

# Save
pathlib.Path("../data/raw").mkdir(parents=True, exist_ok=True)
output.to_csv(OUTPUT_PATH)
print(f"Saved clean yield data → {OUTPUT_PATH}")
print(f"Shape: {output.shape}")
print()
print("Columns exported:")
print("  yield_pct   — daily closing yield level (%)")
print("  daily_change — first difference of yield (bp-equivalent change)")
print("  vol_21d     — 21-day rolling std of daily changes (uncertainty proxy)")
print()
print("This file is ready to be ingested by ingest.py and aligned in preprocess.py.")
output.tail()


## 6. Notes for Technical Report

**Dataset Analysis section — key points to include:**

- Source: Yahoo Finance `^TNX` (CBOE 10-Year Treasury Note Yield Index), daily frequency
- Date range: 2019–2025 captures three distinct regimes: pre-COVID stability, near-zero COVID era, and the 2022–2024 hiking cycle
- Missing values: minimal (US public holidays only); handled via forward-fill to align with ASX trading calendar
- Key sources of uncertainty identified:
  1. **Regime non-stationarity** — the yield distribution differs substantially across macroeconomic regimes
  2. **Heavy tails** — daily changes exhibit excess kurtosis, meaning extreme moves occur more frequently than a Normal distribution predicts
  3. **Volatility clustering** — periods of high yield volatility cluster together (visible in Panel 3 above)

**Justification for Bayesian approach:**

The regime-dependent nature of yield behaviour means a single MLE point estimate
(fit across the full period) will produce confidence intervals that are too narrow
during high-volatility regimes and too wide during stable periods. A Bayesian model
with regime-aware priors — or even a simple hierarchical prior — better represents
the true uncertainty in the parameter estimates. This is the central argument
linking your feature choice to the assignment's core probabilistic framing.
